# AI for Data Analysts with ChatGPT  
## Section 4: Data Cleaning with AI

**Instructor Edition**  
Course: UCI CDE Python for Data Analysis  
Case: BrightMart Retail Analytics

This section teaches students how to use AI to plan data cleaning, while still manually verifying every cleaning decision.

## Learning Objectives

Students will learn to:

1. Identify common data quality issues in a business dataset.
2. Use AI to generate a cleaning plan.
3. Clean duplicates, missing values, inconsistent labels, dates, and outliers.
4. Verify that cleaning did not damage the dataset.
5. Document cleaning decisions clearly.

## Instructor Framing

AI can suggest data-cleaning steps, but it cannot know whether a value is truly valid or invalid without business context.

Tell students:

> AI can help you think through the cleaning workflow, but you must decide what is appropriate for the business problem.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/Colab Notebooks/UCI/UCI_427.62_Python_for_Data_Analysis'
DATA_DIR = f'{BASE_DIR}/05_Datasets'
ASSIGNMENTS_DIR = f'{BASE_DIR}/04_Assignments'
OUTPUTS_DIR = f'{BASE_DIR}/04_Assignments/Outputs'

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

df_raw = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/UCI/UCI_427.62_Python_for_Data_Analysis/07_Capstone_Project/UCI_CDE_Capstone_Package1B/BrightMart_Retail_Dataset.csv")
df_raw.head()

,OrderID,OrderDate,Region,Category,CustomerSegment,Sales,Discount,Profit,Quantity
0,100000,2025-06-15,West,Technology,Consumer,"4,740.37",0.15,710.59,2
1,100001,2025-08-11,South,Office Supplies,Consumer,"2,102.50",0.05,-121.23,10
2,100002,2025-10-27,East,Office Supplies,Consumer,"1,995.47",0.05,-152.93,2
3,100003,2025-10-25,West,Furniture,Corporate,"2,864.60",0.05,181.80,10
4,100004,2025-08-21,East,Furniture,Corporate,"1,820.68",0.05,445.91,10


In [3]:
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/Colab Notebooks/UCI/UCI_427.62_Python_for_Data_Analysis'
DATA_DIR = f'{BASE_DIR}/05_Datasets'
ASSIGNMENTS_DIR = f'{BASE_DIR}/04_Assignments'
OUTPUTS_DIR = f'{BASE_DIR}/04_Assignments/Outputs'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 1 — Inspect the Raw Data

Before asking AI for cleaning code, students should first inspect the dataset.

In [4]:
print("Shape:", df_raw.shape)
print("\nColumns:")
print(df_raw.columns.tolist())

df_raw.info()

Shape: (3535, 9)

Columns:
['OrderID', 'OrderDate', 'Region', 'Category', 'CustomerSegment', 'Sales', 'Discount', 'Profit', 'Quantity']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3535 entries, 0 to 3534
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   OrderID          3535 non-null   int64  
 1   OrderDate        3535 non-null   object 
 2   Region           3513 non-null   object 
 3   Category         3535 non-null   object 
 4   CustomerSegment  3535 non-null   object 
 5   Sales            3535 non-null   float64
 6   Discount         3535 non-null   float64
 7   Profit           3535 non-null   float64
 8   Quantity         3535 non-null   int64  
dtypes: float64(3), int64(2), object(4)
memory usage: 248.7+ KB


In [5]:
raw_quality = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_percent": (df_raw.isna().mean() * 100).round(2),
    "unique_values": df_raw.nunique()
})

raw_quality

,missing_count,missing_percent,unique_values
OrderID,0,0.00,3500
OrderDate,0,0.00,397
Region,22,0.62,5
Category,0,0.00,3
CustomerSegment,0,0.00,3
Sales,0,0.00,3490
Discount,0,0.00,6
Profit,0,0.00,3450
Quantity,0,0.00,12


## Embedded AI Prompt: Cleaning Plan

Use this prompt with ChatGPT:

> I am cleaning a retail sales dataset in pandas.  
> Columns include OrderID, OrderDate, Region, Category, CustomerSegment, Sales, Discount, Profit, and Quantity.  
> Please suggest a practical data-cleaning checklist for duplicates, missing values, inconsistent categories, mixed date formats, negative profit, discounts, and outliers.  
> Do not invent results. Provide verification checks for each cleaning step.

## Expected AI Response

A useful response should mention:

- duplicate checks
- missing-value summary
- string standardization
- date parsing
- numeric range checks
- outlier flags
- verification after cleaning
- documentation of assumptions

If AI suggests deleting all outliers automatically, pause and discuss why that may be risky.

## Step 2 — Remove Exact Duplicates

In [6]:
df = df_raw.copy()

before_rows = len(df)
duplicate_rows = df.duplicated().sum()

df = df.drop_duplicates()
after_rows = len(df)

print("Rows before:", before_rows)
print("Duplicate rows:", duplicate_rows)
print("Rows after:", after_rows)
print("Rows removed:", before_rows - after_rows)

Rows before: 3535
Duplicate rows: 35
Rows after: 3500
Rows removed: 35


### Teaching Checkpoint

Students should explain why exact duplicates may inflate sales, profit, and order counts.

## Step 3 — Standardize Text Fields

In [7]:
print("Region values before cleaning:")
print(df["Region"].value_counts(dropna=False))

Region values before cleaning:
Region
South      878
West       865
East       842
Central    830
west        63
NaN         22
Name: count, dtype: int64


In [8]:
df["Region"] = df["Region"].replace("", np.nan)
df["Region"] = df["Region"].astype("string").str.strip().str.title()

df["Category"] = df["Category"].astype("string").str.strip().str.title()
df["CustomerSegment"] = df["CustomerSegment"].astype("string").str.strip().str.title()

print("Region values after cleaning:")
print(df["Region"].value_counts(dropna=False))

Region values after cleaning:
Region
West       928
South      878
East       842
Central    830
<NA>        22
Name: count, dtype: Int64


### Embedded AI Prompt: Text Standardization

> Explain why `West` and `west` should be standardized before grouping sales by region.  
> Show pandas code using `.str.strip()` and `.str.title()`.

## Step 4 — Parse Dates Safely

In [9]:
print("Sample raw dates:")
print(df_raw["OrderDate"].head(10).tolist())

Sample raw dates:
['2025-06-15', '2025-08-11', '2025-10-27', '2025-10-25', '2025-08-21', '2025-02-07', '2025-06-24', '2025-12-26', '2025-11-09', '2025-07-25']


In [10]:
df["OrderDate"] = pd.to_datetime(df["OrderDate"], errors="coerce")

print("Invalid dates after parsing:", df["OrderDate"].isna().sum())
print("Date range:", df["OrderDate"].min(), "to", df["OrderDate"].max())

Invalid dates after parsing: 32
Date range: 2025-01-01 00:00:00 to 2025-12-31 00:00:00


### Teaching Checkpoint

Students should understand `errors='coerce'`: invalid dates become `NaT`, which can then be counted and investigated.

## Step 5 — Validate Numeric Fields

In [11]:
numeric_checks = df[["Sales", "Discount", "Profit", "Quantity"]].describe()
numeric_checks

,Sales,Discount,Profit,Quantity
count,"3,500.00","3,500.00","3,500.00","3,500.00"
mean,"2,847.71",0.13,208.79,6.58
std,"4,758.82",0.10,379.91,3.43
min,21.32,0.00,-828.02,1.00
25%,"1,231.24",0.05,-23.78,4.00
50%,"2,513.84",0.15,118.69,7.00
75%,"3,759.95",0.20,408.76,10.00
max,"87,500.40",0.30,"1,623.48",12.00


In [12]:
print("Negative sales:", (df["Sales"] < 0).sum())
print("Negative profit:", (df["Profit"] < 0).sum())
print("Discount below 0:", (df["Discount"] < 0).sum())
print("Discount above 1:", (df["Discount"] > 1).sum())
print("Quantity <= 0:", (df["Quantity"] <= 0).sum())

Negative sales: 0
Negative profit: 1042
Discount below 0: 0
Discount above 1: 0
Quantity <= 0: 0


### Instructor Note

Negative profit may be valid. It is a business finding, not automatically a data error.

## Step 6 — Flag Sales Outliers

Outliers should usually be flagged and reviewed, not automatically deleted.

In [13]:
q1 = df["Sales"].quantile(0.25)
q3 = df["Sales"].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr

df["SalesOutlierFlag"] = df["Sales"] > upper_bound

print("Q1:", round(q1, 2))
print("Q3:", round(q3, 2))
print("Upper bound:", round(upper_bound, 2))
print("Outlier count:", df["SalesOutlierFlag"].sum())

df[df["SalesOutlierFlag"]].sort_values("Sales", ascending=False).head(10)

Q1: 1231.24
Q3: 3759.95
Upper bound: 7553.02
Outlier count: 26


,OrderID,OrderDate,Region,Category,CustomerSegment,Sales,Discount,Profit,Quantity,SalesOutlierFlag
2262,102262,2025-07-11,West,Office Supplies,Corporate,"87,500.40",0.00,305.33,11,True
1177,101177,2025-12-01,East,Furniture,Home Office,"81,106.60",0.00,"1,060.80",9,True
1845,101845,2025-02-17,Central,Office Supplies,Consumer,"78,323.60",0.20,-15.11,7,True
2865,102865,2025-11-25,West,Technology,Home Office,"77,845.60",0.05,"1,013.95",12,True
2956,102956,2025-08-02,East,Office Supplies,Consumer,"75,244.40",0.20,-79.21,4,True
3430,103430,2025-02-05,Central,Office Supplies,Consumer,"72,340.20",0.10,147.90,4,True
695,100695,2025-08-11,South,Furniture,Consumer,"66,660.80",0.20,940.70,12,True
210,100210,2025-01-31,West,Office Supplies,Consumer,"63,157.80",0.20,-318.38,5,True
1535,101535,2025-02-08,Central,Technology,Home Office,"62,316.20",0.00,-233.75,2,True
164,100164,2025-06-08,East,Office Supplies,Consumer,"60,847.00",0.30,-30.23,9,True


## Step 7 — Create Analysis-Ready Fields

In [14]:
df["OrderMonth"] = df["OrderDate"].dt.to_period("M").astype(str)
df["ProfitMargin"] = df["Profit"] / df["Sales"]

df[["OrderDate", "OrderMonth", "Sales", "Profit", "ProfitMargin"]].head()

,OrderDate,OrderMonth,Sales,Profit,ProfitMargin
0,2025-06-15,2025-06,"4,740.37",710.59,0.15
1,2025-08-11,2025-08,"2,102.50",-121.23,-0.06
2,2025-10-27,2025-10,"1,995.47",-152.93,-0.08
3,2025-10-25,2025-10,"2,864.60",181.80,0.06
4,2025-08-21,2025-08,"1,820.68",445.91,0.24


## Step 8 — Final Cleaning Verification

In [15]:
cleaning_report = {
    "raw_rows": len(df_raw),
    "clean_rows": len(df),
    "duplicates_remaining": df.duplicated().sum(),
    "missing_region": df["Region"].isna().sum(),
    "invalid_dates": df["OrderDate"].isna().sum(),
    "sales_outliers": df["SalesOutlierFlag"].sum(),
    "total_sales": df["Sales"].sum(),
    "total_profit": df["Profit"].sum()
}

cleaning_report

{'raw_rows': 3535,
 'clean_rows': 3500,
 'duplicates_remaining': np.int64(0),
 'missing_region': np.int64(22),
 'invalid_dates': np.int64(32),
 'sales_outliers': np.int64(26),
 'total_sales': np.float64(9967002.19),
 'total_profit': np.float64(730759.55)}

## Student Reflection Questions

1. Which cleaning step had the greatest impact on the dataset?
2. Which issue should be fixed before executive reporting?
3. Which values should be flagged instead of deleted?
4. How did AI help with the cleaning workflow?
5. What cleaning decision required human judgment?

## Common Student Mistakes

- Deleting outliers without business review.
- Treating negative profit as an error automatically.
- Forgetting to verify duplicates after removal.
- Grouping by region before standardizing region labels.
- Trusting AI-generated cleaning code without checking row counts.

## Section 4 Wrap-Up

A complete cleaning workflow should include:

- inspection
- cleaning plan
- cleaning code
- verification
- documentation
- business interpretation

Next section: Visualization and dashboard creation.

In [16]:
# Optional export for later sections
df.to_csv("BrightMart_Retail_Cleaned_Section4.csv", index=False)
print("Cleaned dataset exported.")

Cleaned dataset exported.
